# Logistic Regression: Theory and Application

Two parts in one notebook:

**Part 1 — Sigmoid function properties** (mathematical + numerical verification)
- Range, monotonicity, limits at ±∞
- Adapting logistic regression for {+1, −1} labels via tanh and threshold modification

**Part 2 — Binary classification: hearing test prediction**
- Dataset: 5,000 participants, features = age + physical score, label = hearing test pass/fail
- EDA with 2D/3D scatterplots and pairplot
- Stratified train/test split + StandardScaler
- Logistic regression with coefficient interpretation and full evaluation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

---
## Part 1 — Sigmoid Function: Mathematical Properties

Let $f(x) = \frac{1}{1 + e^{-x}}$

### 1.1 Range: $f(x) \in (0, 1)$ for all $x \in \mathbb{R}$

For any real $x$, $e^{-x} > 0$, so $1 + e^{-x} > 1$, which gives $\frac{1}{1+e^{-x}} < 1$. Combined with $\frac{1}{1+e^{-x}} > 0$, we have $0 < f(x) < 1$. The bounds 0 and 1 are only approached as limits, never attained.

### 1.2 Monotonicity: $f(x)$ is strictly increasing

Differentiating:
$$f'(x) = \frac{e^{-x}}{(1 + e^{-x})^2}$$

Since $e^{-x} > 0$ and $(1 + e^{-x})^2 > 0$ for all $x$, we have $f'(x) > 0$ everywhere — so $f$ is strictly increasing.

### 1.3 Limits at ±∞

$$\lim_{x \to +\infty} f(x) = \frac{1}{1+0} = 1 \qquad \lim_{x \to -\infty} f(x) = \frac{1}{1+\infty} = 0$$

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

xs = np.linspace(-50, 50, 2000)
ys = sigmoid(xs)

# (a) Range check: all values strictly between 0 and 1
a_ok = np.all((ys > 0) & (ys < 1))

# (b) Monotonicity: all successive differences non-negative
b_ok = np.all(np.diff(ys) >= 0)

# (c) Limit as x → +∞
c_val = sigmoid(100)
c_ok  = np.isclose(c_val, 1.0, atol=1e-12)

# (d) Limit as x → -∞
d_val = sigmoid(-100)
d_ok  = np.isclose(d_val, 0.0, atol=1e-12)

print(f"(a) Range in (0,1)    : {a_ok}")
print(f"(b) Non-decreasing    : {b_ok}")
print(f"(c) Limit +∞ → 1      : {c_ok}  (value = {c_val})")
print(f"(d) Limit -∞ → 0      : {d_ok}  (value = {d_val:.2e})")

Visualising the sigmoid to confirm all four properties:

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(xs, ys, color='steelblue', linewidth=2)
plt.axhline(0, color='gray', linewidth=0.8, linestyle='--')
plt.axhline(1, color='gray', linewidth=0.8, linestyle='--')
plt.axhline(0.5, color='orange', linewidth=0.8, linestyle=':', label='threshold = 0.5')
plt.axvline(0, color='orange', linewidth=0.8, linestyle=':')
plt.title('Sigmoid Function σ(x)')
plt.xlabel('x')
plt.ylabel('σ(x)')
plt.ylim(-0.05, 1.05)
plt.legend()
plt.tight_layout()
plt.show()

---
## Part 1 — Adapting Logistic Regression for {+1, −1} Labels

Standard logistic regression uses labels {0, 1} with threshold 0.5. For {+1, −1} labels, there are two approaches:

### 1.4 Approach A: Change both compression and threshold — use tanh

$$g(z) = \tanh(z/2) = 2\sigma(z) - 1 \in (-1, 1)$$

Natural threshold at 0: predict +1 if $g(z) \geq 0$, else −1. This is equivalent to $\text{sign}(z)$.

In [ ]:
def tanh_compress(z):
    """g(z) = tanh(z/2) = 2*sigmoid(z) - 1, maps z to (-1, 1)"""
    return 2 * sigmoid(z) - 1

def predict_pm1_with_tanh(z):
    return np.where(tanh_compress(z) >= 0, 1, -1)

zs = np.array([-5, -1, 0, 1, 5], dtype=float)
print("z         :", zs)
print("g(z)=tanh :", tanh_compress(zs).round(4))
print("ŷ (tanh)  :", predict_pm1_with_tanh(zs))

### 1.5 Approach B: Keep sigmoid, change only the threshold mapping

Keep $\sigma(z) \in (0,1)$, but instead of mapping $\geq 0.5 \to 1$ and $< 0.5 \to 0$, map to {+1, −1}.

This works because $\sigma(z) \geq 0.5 \iff z \geq 0$ — the decision boundary is unchanged.

In [ ]:
def predict_pm1_with_sigmoid(z):
    return np.where(sigmoid(z) >= 0.5, 1, -1)

print("z              :", zs)
print("σ(z)           :", sigmoid(zs).round(4))
print("ŷ (sigmoid)    :", predict_pm1_with_sigmoid(zs))
print()
print("Both approaches produce identical predictions — same decision boundary at z=0.")
print("Difference: tanh outputs scores in (-1,1); sigmoid outputs probabilities in (0,1).")

---
## Part 2 — Hearing Test Binary Classification

**Research question:** Can age and physical fitness score predict whether someone can hear high-frequency tones?

**Dataset:** 5,000 participants from a hearing study.  
**Features:** `age` (years), `physical_score` (fitness exam score)  
**Label:** `test_result` — 1 if passed hearing test, 0 if not

### 2.1 Load and inspect

In [ ]:
df = pd.read_csv("hearing_test.csv")

print(f"Shape: {df.shape}")
df.head(6)

### 2.2 Exploratory Data Analysis

In [ ]:
df.info()
print()
print(df.describe().round(2))

In [ ]:
mean_age  = df['age'].mean()
q1, q3    = df['physical_score'].quantile([0.25, 0.75])
iqr       = q3 - q1
oldest    = df['age'].max()
label_counts = df['test_result'].value_counts()

print(f"Mean age             : {mean_age:.2f} years")
print(f"IQR of physical_score: {iqr:.2f}")
print(f"Oldest participant   : {oldest:.0f} years")
print(f"\nLabel distribution:")
print(label_counts.to_string())
print(f"\nClass imbalance: {label_counts[1]/len(df)*100:.0f}% passes, {label_counts[0]/len(df)*100:.0f}% fails")
print("→ Will use stratified split to preserve this ratio in train/test sets")

### 2.3 Visualization

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x='age', y='physical_score', hue='test_result', s=25, edgecolor=None)
plt.title('Age vs Physical Score (colored by test result)')
plt.xlabel('Age')
plt.ylabel('Physical Score')
plt.tight_layout()
plt.show()

print("""
Observation: Higher physical_score strongly predicts passing (orange = 1 clusters at top).
Age alone doesn't cleanly separate classes — the groups overlap in the 40–60 range.
The two features together appear to define a diagonal decision boundary.
""")

In [ ]:
sns.pairplot(df, hue='test_result', diag_kind='hist', plot_kws={'s': 18})
plt.suptitle('Pairplot — Hearing Test Dataset', y=1.02)
plt.show()

print("""
Conclusion:
- physical_score is DIRECTLY correlated with passing (higher score → more 1s)
- age is INVERSELY correlated (older → more 0s)
- Diagonal histograms confirm label imbalance (60% pass, 40% fail)
""")

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax  = fig.add_subplot(111, projection='3d')

ax.scatter(
    df['age'], df['physical_score'], df['test_result'],
    c=df['test_result'], cmap='plasma',
    s=18, alpha=0.6, edgecolor='none'
)
ax.set_xlabel('Age')
ax.set_ylabel('Physical Score')
ax.set_zlabel('Test Result')
ax.set_title('3D: Age, Physical Score, Test Result')
plt.tight_layout()
plt.show()

print("""
The 3D view makes the separation clear: two distinct clusters —
yellow (pass) at high physical score, purple (fail) at low score / older age.
This suggests the classes are roughly linearly separable in 2D feature space,
making logistic regression a reasonable model choice.
""")

### 2.4 Preprocessing

Two decisions:
1. **Stratified split** — preserves 60/40 class ratio in both train and test sets
2. **Scaler fit on train only** — prevents test-set statistics from leaking into the model

In [ ]:
X = df[['age', 'physical_score']].values
y = df['test_result'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y   # stratify preserves class balance
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # learn mean/std from training data only
X_test_scaled  = scaler.transform(X_test)        # apply same transform — no fit

print(f"Training samples : {X_train.shape[0]:,}")
print(f"Test samples     : {X_test.shape[0]:,}")
print(f"\nBefore scaling (first 3 rows):\n{X_train[:3]}")
print(f"\nAfter scaling (first 3 rows):\n{X_train_scaled[:3].round(4)}")

### 2.5 Logistic Regression Model

**Coefficient interpretation:** After standardization, coefficients reflect the change in log-odds of passing per standard deviation increase in each feature. A positive coefficient means higher feature value → higher probability of passing.

In [ ]:
logreg = LogisticRegression(random_state=42)
logreg.fit(X_train_scaled, y_train)

print(f"Intercept  : {logreg.intercept_[0]:.4f}")
print(f"β_age      : {logreg.coef_[0][0]:.4f}")
print(f"β_physical : {logreg.coef_[0][1]:.4f}")
print()
print("""
Interpretation:
β_physical ≈ +3.55 — physical score has a strong positive effect on hearing test odds.
β_age ≈ −0.86      — age has a moderate negative effect (older → less likely to pass).
The magnitude ratio (~4:1) confirms physical fitness is the dominant predictor,
consistent with the EDA where age showed weaker class separation than physical score.
""")

### 2.6 Evaluation

In [ ]:
y_pred = logreg.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
cm  = confusion_matrix(y_test, y_pred)
cr  = classification_report(y_test, y_pred, digits=3)

print(f"Test Accuracy: {acc:.3f}\n")
print("Confusion Matrix:")
print(cm)
print()
print("Classification Report:")
print(cr)
print("""
Results:
- 91.7% accuracy on 1,250 test samples
- Class 1 recall (0.947) > class 0 recall (0.872) — model is better at catching passes than fails
- This asymmetry matches the coefficient story: physical_score strongly drives class 1 predictions
- For a hearing screening tool, higher recall on class 1 is generally preferable
  (fewer false negatives = fewer people with good hearing incorrectly flagged as failing)
""")

In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Fail', 'Predicted Pass'],
            yticklabels=['Actual Fail', 'Actual Pass'])
plt.title('Confusion Matrix — Hearing Test Classifier')
plt.tight_layout()
plt.show()